# Automatic Command Execution on Boot

This tutorial is aimed at demonstrating how the main control unit automatically executes specific commands and communicates instructions to the subordinate device each time the system boots. The code blocks in this chapter are for comprehension only and are not executable. They serve to elucidate the automatic processes that the product undertakes upon startup. Should you find the need, these commands are subject to modification or expansion.

## cmd_on_boot() Function
The cmd_on_boot() function, located within the main program of the product, defines a list of commands to be executed at startup. These commands facilitate initial configurations and set up essential operational parameters for the device.

In [ ]:
def cmd_on_boot():
    cmd_list = [
        'base -c {"T":142,"cmd":50}',   # set feedback interval
        'base -c {"T":131,"cmd":1}',    # serial feedback flow on
        'base -c {"T":143,"cmd":0}',    # serial echo off
        'base -c {"T":4,"cmd":0}',      # select the module - 0:None 1:RoArm-M2-S 2:Gimbal
        'base -c {"T":300,"mode":0,"mac":"EF:EF:EF:EF:EF:EF"}',  # the base won't be ctrl by esp-now broadcast cmd, but it can still recv broadcast megs.
        'send -a -b'    # add broadcast mac addr to peer
    ]
    for i in range(0, len(cmd_list)):
        cmdline_ctrl(cmd_list[i])
        cvf.info_update(cmd_list[i], (0,255,255), 0.36)
    set_version(f['base_config']['main_type'], f['base_config']['module_type'])

The product's host computer can control certain functions through command-line instructions, similar to the 'base -c' command above, which is used to directly send the following JSON instructions to the lower-level device via the Raspberry Pi GPIO serial port. Later, we will explain in detail what the default power-on auto-run command means here.

- 'base -c {"T":142,"cmd":50}'
> Used to set the interval for continuous feedback from the lower-level device. The unit for the cmd value is milliseconds. This function is used to reduce the frequency of feedback from the lower-level device, aiming to lessen the computational load on the host computer when processing feedback.
- 'base -c {"T":131,"cmd":1}'
> Enables continuous information feedback from the lower-level device. Once enabled, the host computer does not need to query the lower-level device one question at a time. Normally, the lower-level device has this function turned on by default, but here we send the command again to ensure it is enabled, for safety.
- 'base -c {"T":143,"cmd":0}'
> Disables serial command echo. This way, when the host computer sends commands to the lower-level device, the device will not echo the received commands back to the host, avoiding unnecessary information processing.
- 'base -c {"T":4,"cmd":0}'
> Sets the type of external module. When cmd equals 0, it means no external module; 1 is a robotic arm; 2 is a gimbal. If your product does not have a gimbal or robotic arm installed, you need to change this value to 0. To avoid excessively long feedback data, it is recommended to set it to 0.
- 'base -c {"T":300,"mode":0,"mac":"EF:EF:EF:EF:EF:EF"}'
> Prevents the chassis from being controlled by ESP-NOW broadcasts from other devices, except for the device with the specified MAC address. You can freely generate a MAC address yourself or use the MAC address of your own ESP32 remote controller.
- 'send -a -b'
> Adds the broadcast address (FF:FF:FF:FF:FF:FF) to the peer list, allowing you to directly send broadcast information to other devices in the future via broadcast signals.

You can learn about other host computer command line instructions in the following WEB command line application chapters.